# Flower image classification with transfer learning

This project takes the ideas from the foundation notebooks into a complete image-classification workflow. The task is to identify one of 102 flower categories from an image. Training a convolutional network from random initialization would require more data and compute than this project needs, so the experiment starts with a network pretrained on ImageNet and replaces its final classifier.

The convolutional layers provide a fixed visual representation. Only the new classifier is trained on the flower data. This makes the experiment a useful example of transfer learning: the earlier representation is reused, while the final decision rule is fitted to a new label space.

The dataset is not downloaded by this notebook. Download it manually using the instructions in the project README, then run the cells from top to bottom.


## Dataset and experimental question

The Oxford 102 Flowers dataset is already split into `train`, `valid`, and `test` directories. Each directory contains one subdirectory per class, which makes it compatible with `torchvision.datasets.ImageFolder`. The class names are numeric directory names; `cat_to_name.json` maps those ids to flower names.

The main question is not whether a large pretrained model can memorize the images. It is whether a small task-specific classifier can use pretrained visual features to classify a new set of categories, and whether validation performance gives a useful estimate of the final test result.


In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

seed = 23
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

data_dir = Path('flower_data')
if not (data_dir / 'train').exists():
    raise FileNotFoundError(
        'flower_data/train is missing. Download and extract the dataset using projects/flower_classifier/README.md.'
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 64
num_classes = 102
print('device:', device)


## Preprocessing and data loaders

The pretrained ImageNet model expects 224×224 RGB images normalized with the ImageNet channel statistics. Training images receive random crops and horizontal flips so that the classifier does not depend on one exact framing. Validation and test images use a deterministic resize and center crop; their transformations must not introduce random changes when performance is being measured.


In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

image_datasets = {
    'train': datasets.ImageFolder(data_dir / 'train', transform=train_transform),
    'valid': datasets.ImageFolder(data_dir / 'valid', transform=eval_transform),
    'test': datasets.ImageFolder(data_dir / 'test', transform=eval_transform),
}
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
    'valid': DataLoader(image_datasets['valid'], batch_size=batch_size),
    'test': DataLoader(image_datasets['test'], batch_size=batch_size),
}

with open('cat_to_name.json') as file:
    cat_to_name = json.load(file)

print({split: len(dataset) for split, dataset in image_datasets.items()})
print('classes:', len(image_datasets['train'].classes))


## Inspecting the label distribution

Before training, check how many examples each split contains and whether the class counts are roughly comparable. A highly uneven class distribution changes how useful accuracy is as a summary. The plot shows the twenty largest training classes; the complete mapping remains in `cat_to_name.json`.


In [ ]:
train_counts = pd.Series(image_datasets['train'].targets).value_counts().sort_index()
count_data = pd.DataFrame({
    'class_id': train_counts.index.astype(str),
    'class_name': [cat_to_name[str(i)] for i in train_counts.index],
    'images': train_counts.values,
}).sort_values('images', ascending=False).head(20)

sns.set_theme(style='whitegrid', context='notebook')
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=count_data, y='class_name', x='images', color='tab:blue', ax=ax)
ax.set(title='Twenty largest training classes', xlabel='training images', ylabel='')
fig.tight_layout()
plt.show()


## Replacing the final classifier

VGG16 was trained on ImageNet, whose final layer has 1,000 outputs. Those outputs are not the flower labels, so the final classifier is discarded. The convolutional feature extractor is frozen, and a new classifier maps its 25,088-dimensional flattened representation to 102 flower classes.

The loss is negative log likelihood applied to log probabilities. Adam updates only the new classifier parameters. Freezing the feature extractor keeps the experiment small and isolates the transfer-learning decision; fine-tuning all layers would be a separate experiment with a different compute and regularization profile.


In [ ]:
weights = models.VGG16_Weights.IMAGENET1K_V1
model = models.vgg16(weights=weights)
for parameter in model.features.parameters():
    parameter.requires_grad = False

model.classifier = nn.Sequential(
    nn.Linear(25088, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, num_classes),
    nn.LogSoftmax(dim=1),
)
model = model.to(device)

criterion = nn.NLLLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
print(f'trainable parameters: {trainable:,} / {total:,}')


## Training and validation

Each epoch makes one pass through the training set, then evaluates the unchanged model on the validation set. The training loss measures optimization on augmented training images. Validation loss and accuracy measure the current classifier on deterministic images it did not update from. Keeping the two phases separate is what makes the validation curve useful for choosing the number of epochs.


In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    correct = 0
    examples = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            log_probs = model(images)
            loss = criterion(log_probs, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (log_probs.argmax(dim=1) == labels).sum().item()
        examples += images.size(0)
    return total_loss / examples, correct / examples


history = []
epochs = 5
for epoch in range(epochs):
    train_loss, train_accuracy = run_epoch(model, dataloaders['train'], optimizer)
    valid_loss, valid_accuracy = run_epoch(model, dataloaders['valid'])
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'valid_loss': valid_loss,
        'train_accuracy': train_accuracy,
        'valid_accuracy': valid_accuracy,
    })
    print(
        f"epoch {epoch + 1}: train loss={train_loss:.3f}, "
        f"valid loss={valid_loss:.3f}, valid accuracy={valid_accuracy:.3f}"
    )

history = pd.DataFrame(history)


## Reading the learning curves

The loss and accuracy curves should be read together. A falling training loss with a flat or worsening validation loss indicates that additional fitting is no longer improving generalization. The validation set is the basis for selecting a checkpoint; the test set remains untouched until the end.


In [ ]:
long_history = history.melt(
    id_vars='epoch',
    value_vars=['train_loss', 'valid_loss', 'train_accuracy', 'valid_accuracy'],
    var_name='metric',
    value_name='value',
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=long_history[long_history.metric.str.contains('loss')], x='epoch', y='value', hue='metric', marker='o', ax=axes[0])
sns.lineplot(data=long_history[long_history.metric.str.contains('accuracy')], x='epoch', y='value', hue='metric', marker='o', ax=axes[1])
axes[0].set(title='Loss', ylabel='negative log likelihood')
axes[1].set(title='Accuracy', ylabel='fraction correct')
fig.tight_layout()
plt.show()


## Test evaluation and top-k accuracy

The test set is evaluated once after the training choices have been made. Top-1 accuracy asks whether the first prediction is correct. Top-5 accuracy asks whether the correct flower appears among the five highest-probability classes, which is useful when visually similar species are difficult to separate. Neither metric explains individual failures, so the final cell also keeps the predictions available for inspection.


In [ ]:
def evaluate(model, loader, topk=(1, 5)):
    model.eval()
    hits = {k: 0 for k in topk}
    total = 0
    rows = []
    with torch.no_grad():
        for images, labels in loader:
            log_probs = model(images.to(device))
            probabilities = log_probs.exp()
            values, indices = probabilities.topk(max(topk), dim=1)
            labels = labels.to(device)
            for k in topk:
                hits[k] += (indices[:, :k] == labels[:, None]).any(dim=1).sum().item()
            rows.extend(zip(labels.cpu().numpy(), indices.cpu().numpy(), values.cpu().numpy()))
            total += labels.size(0)
    return {f'top_{k}_accuracy': hits[k] / total for k in topk}, rows


test_metrics, test_rows = evaluate(model, dataloaders['test'])
print(test_metrics)


## Saving the model and inspecting one prediction

The checkpoint stores the architecture choice, classifier shape, model parameters, and the `class_to_idx` mapping created by `ImageFolder`. The mapping is part of the model artifact: without it, the numeric output index cannot be translated back to the correct flower category.


In [ ]:
checkpoint_path = Path('checkpoints') / 'flower_vgg16_classifier.pth'
checkpoint_path.parent.mkdir(exist_ok=True)
torch.save({
    'arch': 'vgg16',
    'hidden_units': 512,
    'classifier': model.classifier,
    'state_dict': model.state_dict(),
    'class_to_idx': image_datasets['train'].class_to_idx,
}, checkpoint_path)
print('saved:', checkpoint_path)

image_path = next((data_dir / 'test').glob('*/*.jpg'))
image = Image.open(image_path).convert('RGB')
input_tensor = eval_transform(image).unsqueeze(0).to(device)
with torch.no_grad():
    probabilities = model(input_tensor).exp().squeeze(0)
values, indices = probabilities.topk(5)
predictions = pd.DataFrame({
    'flower': [cat_to_name[str(image_datasets['test'].classes[i])] for i in indices.cpu().numpy()],
    'probability': values.cpu().numpy(),
})
print('image:', image_path)
print(predictions.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image)
ax.axis('off')
ax.set_title(predictions.iloc[0]['flower'])
plt.show()


## What this experiment establishes

This is a transfer-learning baseline, not a claim that VGG16 is the best model for the dataset. The experiment establishes a complete path from labeled image directories to preprocessing, frozen pretrained features, a task-specific classifier, validation-based training, held-out testing, checkpointing, and inference.

The main limitations are the fixed ImageNet representation, the small classifier experiment, and the dependence on the dataset split. A useful next comparison would fine-tune the last convolutional block or replace VGG16 with a smaller modern backbone, keeping the test set and evaluation code unchanged.
